<a href="https://colab.research.google.com/github/Avni2000/cs639-final-project/blob/main/probe-baseline/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install pandas
%pip install transformers
%pip install torch
%pip install datasets
# restart the kernel after installing in your virtual environment to ensure the new packages are loaded

In [2]:
import pandas as pd

## Grab our dataset of historical AIME problems

In [6]:

from datasets import load_dataset

ds = load_dataset("gneubig/aime-1983-2024")

# get all features
filtered = ds['train']

# Save to csv
filtered.to_csv('aime_historical.csv')

df = pd.read_csv('./aime_historical.csv')
display(df.tail(50))

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

,ID,Year,Problem Number,Question,Answer,Part
883,2022-II-9,2022,9,Let $\ell_A$ and $\ell_B$ be two distinct para...,244,II
884,2022-II-10,2022,10,Find the remainder when \[\binom{\binom{3}{2}}...,4,II
885,2022-II-11,2022,11,Let $ABCD$ be a convex quadrilateral with $AB=...,180,II
886,2022-II-12,2022,12,"Let $a, b, x,$ and $y$ be real numbers with $a...",23,II
887,2022-II-13,2022,13,There is a polynomial $P(x)$ with integer coef...,220,II
888,2022-II-14,2022,14,"For positive integers $a$ , $b$ , and $c$ with...",188,II
889,2022-II-15,2022,15,Two externally tangent circles $\omega_1$ and ...,140,II
890,2023-I-1,2023,1,Five men and nine women stand equally spaced a...,191,I
891,2023-I-2,2023,2,Positive real numbers $b \not= 1$ and $n$ sati...,881,I
892,2023-I-3,2023,3,"A plane contains $40$ lines, no $2$ of which a...",607,I


## Testing

Let's do a small test to see if our thinking traces works, we're calling HF correctly, etc.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"  # pull from HuggingFace
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, output_hidden_states=True)

inputs = tokenizer("Why is the sky blue?", return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)

# outputs.hidden_states is a tuple: one tensor per layer
# each tensor shape: (batch, seq_len, hidden_dim)
hidden_states = outputs.hidden_states

for layer_idx, layer_hs in enumerate(hidden_states):
    print(f"Layer {layer_idx}: {layer_hs.shape}")
    # e.g. Layer 0: torch.Size([1, 8, 2048])
    #               batch ^ ^ tokens ^ ^ ^ hidden_dim

# store as list of tensors per token across all layers:
# shape: [num_layers][batch, token, hidden_dim]
all_layers = [hs.squeeze(0) for hs in hidden_states]  # drop batch dim
# all_layers[layer][token] → hidden vector for that token at that layer